# TP06 : Réseaux de Neurones 1 et Révision Sklearn


### Modalités de rendu

1. Chaque étudiant doit rendre un travail individuel.
2. Renommez ce fichier selon la convention : `TP06_Nom_Prenom.ipynb`.
3. Le rendu s'effectuera via le lien de dépôt communiqué par votre enseignant.
4. Assurez-vous que votre code s'exécute sans erreur (Menu : Kernel > Restart & Run All).

### Objectifs de la séance

Ce TP a pour objectif d'apprendre à créer des réseaux de neurones avec sklearn, ici pour résoudre un problème de régression, ainsi que de revoir les routines de sklearn pour être prêt pour le projet.

### Documentation utile

- [Scikit-Learn MLPRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html#sklearn.neural_network.MLPRegressor)
- [sklearn Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html)
- [sklearn GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)

## Organisation du TP

Ce TP contient 3 parties :

1. **Régression par Réseaux de Neurones Artificiels**
2. **Optimisation des hyperparamètres**
3. **Récapitulatif des routines de sklearn**

## Imports globaux

Exécutez cette cellule avant de commencer.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score


import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

## Chargement du jeu de données Combined Cycle Power Plant

CCPP contient 9568 lignes de données collectés d'une centrale électrique sur une période de 6 ans (2006-2011), lorsqu'elle fonctionnait encore à plein régime.  

In [ ]:
# Chargement CCPP — cette cellule est fournie, exécutez-la simplement
print("Chargement des données...")
ccpp_df = pd.read_csv(".\CCPP\Power_Plant_Data.csv")

Visualisez le jeu de donnée grâce à l'attribut `info` et la méthode `head()`.

Vérifiez également qu'il n'y a pas de données manquantes dans votre jeu de données.

In [ ]:
print(ccpp_df....)
print(ccpp_df....())

ccpp_df....().values.any()

Ce jeu de données possède 5 attributs :
- La Température (AT) en Celcius
- L'Exhaust Vacuum (V) en millimètre de mercure (unité de pression)
- La Pression Ambiante (AP) en millibar
- L'Humidité Relative (RH) en pourcentage
- La Production d'énergie électrique nette horaire (PE) en MegaWatt

L'objectif dans la partie 1 va être de créer un réseau de neurones capable de prédire la production d'énergie a partie des 4 autres caractéristiques.

---

# Partie 1 — Régression par Réseaux de Neurones Artificiels

## Rappel théorique

Un neurone artificiel est une fonction mathématique prenant en entrée un vecteur $x$ et retournant une valeur $y$ tel que :

$$y=\sigma(x*w+b)$$

avec
- $w$ un vecteur de coefficients s'appliquant aux caractéristiques d'entrées, appelés **poids**
- $b$ un scalaire appelé **biais**
- $\sigma$ un fonction servant à donner le signe de la quantité calculée (vous avez vu en cours la fonction de Heavyside et la fonction logistique)


Un Réseau de neurones est un algorithme utilisant des couches de neurones, où chaque neurone d'une couche donnée va recevoir en entrée les valeurs des neurones de la couche précédente.

L'algorithme de descente de gradient permet au modèle d'ajuster les poids et biais des neurones pendant la phase d'entrainement du modèle.

## Étape 1 — Standardisation

Avant tout travail sur un algorithme d'apprentissage, il est crucial de **centrer** les données pour éviter de donner artificiellement trop d'importance à certains attributs.

In [ ]:
# Séparation train/test avec un ratio de 80%/20%
X=ccpp_df[[...]]
y=ccpp_df[[...]]

X_train, X_test, y_train, y_test = train_test_split(
    ..., ..., test_size=..., random_state=42
)

scaler1 = StandardScaler()
X_train_sc = scaler1....(...)   # On fit notre scaler et on transforme ces donnée
X_test_sc  = scaler1....(...)   # transform seulement — pas de fit !

# On créer un scaler différent pour les données que l'on cherche à prédire.
# On a envie de centrer nos données pour notre modèle, 
# mais on a besoin de pouvoir les dé-transformer après pour avoir les vrais valeurs.
scaler2 = StandardScaler()
y_train_sc = scaler2....(...)   # On fit notre scaler et on transforme ces donnée
y_test_sc  = scaler2....(...)   # transform seulement — pas de fit !


## Étape 2 — Créer un premier réseau de neurones

Sklearn permet de créer des Perceptrons Multicouches pour de la classification (**MLPClassifier**) et de la régression (**MLPRegressor**), celui qui nous intéresse pour ce TP.

On souhaite que ce réseau de neurones ait :
- 2 couches de neurones cachés comprenant 10 neurones chacuns,
- la fonction d'activation de ces neurones soit la fonction "ReLU",
- la fonction d'optimisation des poids soit celle de la descente de gradient stochastique.

On ne se préoccupe pas pour l'instant de savoir si ces hyperparamètres sont optimaux, on cherche juste à se familiariser avec le modèle.

In [ ]:
model = ...(hidden_layer_sizes=...,activation=...,solver=...,random_state=42)

Entrainons maintenant notre modèle.

Attention, notre problème étant un problème de régression, nous ne pouvons évaluer la performance notre modèle avec des métriques faîtes pour évaluer des modèles de classification (comme `accuracy_score()`).
Plusieurs métriques existent pour évaluer des modèles de régression. Nous utiliserons ici une que vous avez vu en cours, le coefficient de détermination $R^2$ (`r2_score()`).

Pour rappel, $R^2=1-\frac{SS_{res}}{SS_{tot}}=1-\frac{\sum\limits_{i=1}\limits^{n}(y^{(i)}-\hat{y}^{(i)})^2}{\sum\limits_{i=1}\limits^{n}(y^{(i)}-\bar{y})^2}$, avec :

- $SS_{res}$=somme des carrés des résidus,
- $SS_{tot}$=somme des carrés totale,
- $y^{(i)}$ les valeurs réelles,
- $\hat{y}^{(i)}$ les valeurs prédites,
- $\bar{y}$ la moyenne des valeures réelles.

Plus le coefficient est proche de 1, plus précises sont nos prédictions.

In [ ]:
model.fit(...,...)
y_pred_sc=model.predict(...)

print( "Le coefficient de détermination des données scalées prédites par notre modèle est de :",
      ...)

# On veut regarder le résultat avec nos données prédites unscale.
# On doit donc faire la transformée inverse de nos données prédites
y_pred_unsc=scaler2....(np.reshape(y_pred_sc,(-1,1)))

print( "Le coefficient de détermination des données scalées prédites par notre modèle est de :",
      ...)



On voit que la valeur du coefficient de détermination ne change pas quand on unscale les données. Attention, ça n'est pas forcément le cas pour toutes les métriques utilisées (le Mean Squared Error en est un bon exemple).

Visualisons maintenant nos résultats. Nous allons tracer la distribution des résidus avec un histogramme, et faire un scatterplot des résultats prédits contre résultats réels.

In [ ]:
residuals = ... - ...

# Plot 1: Residuals Distribution.
plt.subplot(1,2,1)
sns.histplot(residuals, bins=30, kde=True, color="blue")
plt.axvline(x=0, color='red', linestyle='--')
plt.title("Residuals Distribution")
plt.xlabel("Residuals (y_vrai - y_predit)")
plt.ylabel("Frequency")

# Plot 2: Regression Fit (Actual vs Predicted).
plt.subplot(1,2,2)
sns.scatterplot(x=np.concatenate(np.array(y_test)), y=np.concatenate(y_pred_unsc))
#plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='red', linestyle='--')  # Perfect fit line
plt.title("Regression Fit: Vrai vs Predit")
plt.xlabel("Vrai valeurs (en MW)")
plt.ylabel("Valeurs prédites (en MW)")


---

# Partie 2 — Optimisation des hyperparamètres par Validation croisée

## Rappel théorique

En apprentissage automatique il est commun d'utiliser plusieurs échantillons de données :
- Données d'apprentissage : pour l'entraînement du modèle.
- Données de validation : pour tester différents espaces des hypothèses H.
- Données de test : pour l'évaluation finale et indépendante du modèle.

Un échantillon de validation fixe présente des avantages dès qu'il s'agit de comparer des estimateurs différents sur les mêmes données. Mais il est aussi courant de générer les données de validations à partir des données d'apprentissage.

Il existe plusieurs méthode de **validation croisée**, mais nous nous intéresserons surtout ici à la *Validation croisée à K plis** (**K Fold** en anglais).

La méthode de validation croisée à K plis consiste à partitionner aléatoirement les données en K sous-ensembles (appelés plis ou blocs) de tailles égales.

Soit $\{I_1,...,I_K\}$ l'ensemble des plis. Pour chaque plis $i$, on entraine notre modèle sur l'ensemble $I_1\cup...\cup I_K\backslash I_i$, et on évalue le modèle ainsi entrainé sur $I_i$. La performance d'un set d'hyperparamètre donnée est alors la moyenne des performances pour chaque plis.

Pour nous aider à comprendre, essayons d'implémenter cela.
Nous souhaitons tester les hyperparamètres suivant pour notre modèle :
- 1, 2 ou 3 couches cachées,
- les couches cachées contiennent toute 10, 20 ou 30 neurones,
- la fonction d'activation est soit ReLu, soit TanH, soit Logistic,
- la méthode de descente de gradient est soit "sgd", soit "adam".

In [ ]:
num_layers=[...]
layer_size=[...]
activation=[...]
solver=[...]

Nous souhaitons séparer notre jeu de données d'entrainement en 5 grâce à la `KFold()`:

In [ ]:
kf=...(n_splits=...,shuffle=True,random_state=42)

#On visualise les sous-jeux de données
for i, (train_index, test_index) in enumerate(kf.split(X_train_sc,y_train_sc)):
    print(f"Fold {i}:")
    print(f"  Train: index={train_index}")
    print(f"  Test:  index={test_index}")

Maintenant, on souhaite tester notre modèle pour chaque hyperparamètre. 

In [ ]:
for nl in ... : # Pour chaque quantité de couches cachées
    for sl in ...: # Pour chaque taille de couche
        for a in activation: # Pour chaque fonction d'activation
            for s in ...: # Pour chaque solver
                hls=np.ones(nl,dtype=np.int64)*sl
                model2 = MLPRegressor(hidden_layer_sizes=...,activation=...,solver=...,random_state=42)
                score=[]
                for i, (train_index, test_index) in enumerate(kf.split(X_train_sc,y_train_sc)):
                    model2.fit(...[...],...[...])
                    y_pred=model2.predict(...[...])
                    score.append(r2_score(y_true=...[...],y_pred=...))
                print(
                    "num_layer : ",nl,
                    "\nlayer size : ",sl,
                    "\nsolver : ",s,
                "score = ", np.mean(score)
            )
            

Ce travail peut être assez fastidieux. Heureusement, il existe des fonctions déja implémentées dans *sklearn* qui vont nous permettre de grandement simplifier l'écriture du code.

La fonction `GridSearchCV()` permet d'effectuer les tests de tout les hyperparamètres voulu pour n'importe quel modèle, et d'obtenir les scores calculés pour chaque combinaison. On veut séparer notre jeu d'entrainement en 5 et que la fonction de score soit **r2_score**.

In [ ]:
mlpr=MLPRegressor()

hyperparam={
    'hidden_layer_sizes':[[10],[20],[30],[10,10],[20,20],[30,30],[10,10,10],[20,20,20],[30,30,30]],
    'solver':["adam","sgd"],
    'activation':["relu","tanh","logistic"]
}

gscv=GridSearchCV(...,...,cv=...,scoring=...)

gscv.fit(...,...)

print(gscv.cv_results_)

In [ ]:
cv_res=np.dstack((np.array(gscv.cv_results_["params"]),
                  gscv.cv_results_["mean_test_score"],
                  gscv.cv_results_["rank_test_score"]))[0]

# Hyperparamètre ordonnées du plus au moins performant
print(cv_res[cv_res[:,2].argsort()])

print("Le meilleur set d'hyperparamètres parmis ceux testés est : ",
      cv_res[cv_res[:,2].argsort()][0][0])

### 📝 Observations

*Quels choix d'hyperparamètres semblent améliorer le plus notre modèle ?*


# Partie 3 — Récapitulatif des routines de sklearn

Maintenant, nous allons révoir une dernière fois tout ce qui a été fait.
- On doit utiliser **train_test_split** pour séparer notre jeu de données en 2, des données d'entrainement et des données de test.
- Ensuite, on va utiliser une **Pipeline()** pour effectuer des transformations successives de nos données jusqu'à entrainer notre modèle (ici, on mettra dans la pipeline un scaler puis notre perceptron multiouche).
- Nous allons faire une validation croisée pour trouver les meilleurs hyperparamètres à choisir.
- Enfin, nous entraineront notre modèle optimisé sur les données d'entrainement, et le testeront sur des données test.

N'hésitez pas à essayer d'autres hyperparamètres que ceux proposés pour essayer d'avoir d'encore meilleurs résultats !

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    ..., ..., test_size=0.2, random_state=42
)

pipeline = Pipeline([
    ('scaler', ...()),
    ('mlpreg', ...(random_state=42))
])

# Attention ! Quand on code nos hyperparamètres à optimiser en passant par une Pipeline, 
# il ne faut pas oublier de formater le nom de l'hyperparamètre en rajoutant le nom de l'étape suivi de 
# deux underscore.
hyperparam = {
    'mlpreg__...':[[10],[20],[30],[10,10],[20,20],[30,30],[10,10,10],[20,20,20],[30,30,30]],
    'mlpreg__...':["adam","sgd"],
    'mlpreg__...':["relu","tanh","logistic"]
}

gscv=GridSearchCV(...,...,cv=...,scoring=...)

gscv.fit(...,...)

cv_opti=np.dstack((np.array(gscv.cv_results_["params"]),
                   gscv.cv_results_["mean_test_score"],
                   gscv.cv_results_["rank_test_score"]))[0]
#On garde les hyperparamètres donnant les meilleurs performances
param_opti=cv_opti[cv_opti[:,2].argsort()][0][0]

final_model = Pipeline([
    ('scaler', ...()),
    (('mlpreg', ...(hidden_layer_sizes=param_opti[...],
                    activation=param_opti[...],
                    solver=param_opti[...],
                    random_state=42)))
])

final_model.fit(...,...)

y_pred=final_model.predict(...)
print("Le coefficient de détermination des données scalées prédites par notre modèle est de :",
      ...)
